# SICD Viewer & Metadata Inspection

**Source**: `grdl/example/IO/sar/view_sicd.py`

## Learning Objectives

- Quick SICD metadata inspection
- Chip extraction and display
- Metadata summary for mission planning

## Data Setup

**Path Configuration**: Set path to a SICD .nitf file.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
# Configuration
SICD_PATH = '/path/to/your/sicd_file.nitf'
CHIP_SIZE = 1024  # Display center chip

print(f"SICD file: {SICD_PATH}")
print(f"Chip size: {CHIP_SIZE}x{CHIP_SIZE}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from grdl.IO import get_reader
from grdl.contrast import MangisDensity

In [ ]:
# Load SICD and extract metadata
with get_reader('sicd', SICD_PATH) as reader:
    metadata = reader.metadata
    rows, cols = reader.get_shape()
    
    print(f"\nImage shape: {rows} x {cols}")
    print(f"Collection: {metadata.collection_info.collector_name if metadata.collection_info else 'N/A'}")
    print(f"Polarization: {metadata.image_data.polarization if metadata.image_data else 'N/A'}")
    print(f"Center frequency: {metadata.radar_collection.tx_frequency_range.center / 1e9:.2f} GHz" 
          if metadata.radar_collection and metadata.radar_collection.tx_frequency_range else "N/A")
    
    # Extract center chip
    r0 = max(0, rows // 2 - CHIP_SIZE // 2)
    c0 = max(0, cols // 2 - CHIP_SIZE // 2)
    chip = reader.read_chip(r0, r0 + CHIP_SIZE, c0, c0 + CHIP_SIZE)
    
    print(f"\nChip extracted: [{r0}:{r0+CHIP_SIZE}, {c0}:{c0+CHIP_SIZE}]")

In [ ]:
# Apply Mangis Density contrast
contrast = MangisDensity()
magnitude = np.abs(chip)
display = contrast.apply(magnitude)

print(f"\nMagnitude range: [{magnitude.min():.2e}, {magnitude.max():.2e}]")
print(f"Display range: [{display.min():.3f}, {display.max():.3f}]")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(display, cmap='gray')
ax.set_title(f'SICD Chip: {CHIP_SIZE}x{CHIP_SIZE}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cleanup memory
import gc

del chip, metadata, magnitude, display
gc.collect()
print("✓ Memory cleanup complete")

## Summary

**GRDL patterns**:
- ✅ `get_reader('sicd', path)` — factory pattern
- ✅ `read_chip()` — efficient sub-region extraction
- ✅ `MangisDensity()` — SAR contrast enhancement